# Modul 6 — ML Model Endpoints (Iris Sample)

Notebook companion untuk [`06-model-endpoints.md`](../06-model-endpoints.md).

**Tujuan**: Train & register sample model **Iris** yang dijamin **endpoint-compatible** (column-based schema, bukan tensor) supaya kamu bisa langsung mengaktifkan **ML Model Endpoint** di Fabric UI.

## Alur notebook

| Cell | Aksi |
|---|---|
| 1-2 | Setup library + load Iris dataset built-in sklearn |
| 3 | Train + log + register model dengan signature column-based |
| 4 | Verifikasi schema (harus `ColSpec`, **bukan** `Tensor`) |
| 5 | (Optional) Test prediksi lokal sebelum aktivasi endpoint |
| 6-7 | (Optional) Call REST API endpoint setelah Activate dari UI |

> ⚠️ **Aturan emas agar endpoint enabled**:
> 1. `X` harus `pd.DataFrame` — **bukan** `np.ndarray`
> 2. `input_example=X_train.head(N)` — **bukan** `X_train.values[:N]`
> 3. Hindari pipeline yang outputnya ndarray

## 1. Import library

Semua library di bawah sudah pre-installed di runtime Fabric (Spark / Python pool). Tidak perlu `pip install`.

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from mlflow.models.signature import infer_signature

print(f"MLflow version : {mlflow.__version__}")
print(f"pandas version : {pd.__version__}")

## 2. Load dataset Iris (publik, built-in sklearn)

Dataset ini sudah ter-bundle di `scikit-learn` — **tidak perlu download** apapun.

- 150 baris × 4 fitur numerik
- Target: 3 kelas bunga Iris (setosa / versicolor / virginica)

> 🔑 Parameter `as_frame=True` menghasilkan `pandas.DataFrame` — **kunci** agar MLflow nanti meng-infer schema sebagai **column-based** (`ColSpec`), bukan tensor.

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data           # ⚠️ pandas DataFrame
y = iris.target         # pandas Series

print(f"X type   : {type(X).__name__}")
print(f"X shape  : {X.shape}")
print(f"Columns  : {list(X.columns)}")
print(f"Classes  : {dict(enumerate(iris.target_names))}")

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## 3. Train + log + register model

Parameter penting untuk `mlflow.sklearn.log_model`:

| Parameter | Nilai | Kenapa |
|---|---|---|
| `signature=infer_signature(X_train, preds)` | DataFrame in/out | Hasilkan `ColSpec` (column-based) |
| `input_example=X_train.head(3)` | DataFrame, bukan `.values` | Sample untuk MLflow UI + memperkuat schema column-based |
| `registered_model_name="iris_sample_model"` | string | Otomatis register ke Fabric workspace |

In [ ]:
mlflow.set_experiment("iris-endpoint-demo")

with mlflow.start_run() as run:
    # Train
    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_train, y_train)

    # Evaluate
    accuracy = model.score(X_test, y_test)
    mlflow.log_param("n_estimators", 50)
    mlflow.log_metric("accuracy", accuracy)

    # Signature column-based (KUNCI agar endpoint enabled)
    # ⚠️ Output HARUS pd.Series/DataFrame — kalau ndarray → schema jadi Tensor!
    preds_train = pd.Series(model.predict(X_train), name="prediction")
    signature = infer_signature(X_train, preds_train)

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(3),     # ⬅️ DataFrame
        registered_model_name="iris_sample_model",
    )

    print(f"✅ Run ID  : {run.info.run_id}")
    print(f"✅ Accuracy: {accuracy:.4f}")
    print(f"✅ Model registered as: iris_sample_model")

## 4. Verifikasi schema (harus `ColSpec`, bukan `Tensor`)

Output yang **benar** harus mengandung baris-baris seperti:

```
['sepal length (cm)': double (required), 'sepal width (cm)': double (required), ...]
```

Kalau muncul `Tensor(...)` → kembali ke Cell 2-3 dan pastikan `X` adalah DataFrame.

In [ ]:
client = mlflow.tracking.MlflowClient()

# Ambil versi terbaru
versions = client.search_model_versions("name='iris_sample_model'")
latest = max(versions, key=lambda v: int(v.version))

print(f"Latest version: {latest.version}")
print(f"Source URI    : {latest.source}\n")

loaded = mlflow.pyfunc.load_model(latest.source)
input_schema = loaded.metadata.get_input_schema()
output_schema = loaded.metadata.get_output_schema()

print("Input schema:")
print(input_schema)
print("\nOutput schema:")
print(output_schema)

# Cek tipe schema (input + output)
combined = f"{input_schema}|{output_schema}"
if "Tensor" in combined:
    print("\n❌ TENSOR schema terdeteksi — endpoint TIDAK BISA diaktifkan!")
    print("   Pastikan X = pd.DataFrame DAN output predict dibungkus pd.Series.")
elif input_schema is None or output_schema is None:
    print("\n❌ Schema kosong — endpoint TIDAK BISA diaktifkan!")
else:
    print("\n✅ Column-based schema (input + output) — endpoint READY di Fabric UI!")

## 5. (Optional) Test prediksi lokal

Sebelum aktivasi endpoint di Fabric UI, kita bisa test prediksi langsung dari MLflow model object — untuk pastikan model bisa serve prediksi dengan format DataFrame.

In [ ]:
sample_input = pd.DataFrame(
    {
        "sepal length (cm)": [5.1, 6.7],
        "sepal width (cm)":  [3.5, 3.0],
        "petal length (cm)": [1.4, 5.2],
        "petal width (cm)":  [0.2, 2.3],
    }
)

predictions = loaded.predict(sample_input)
class_names = iris.target_names

print("Input:")
print(sample_input, "\n")
print(f"Predictions: {predictions}")
print(f"Class names: {[class_names[p] for p in predictions]}")

## 6. Aktivasi endpoint di Fabric UI

Sekarang buka Fabric workspace dan ikuti:

1. Workspace → klik ML model **`iris_sample_model`**
2. Buka **Version** terbaru
3. Klik **Activate version endpoint** di ribbon (sekarang **enabled** ✅)
4. Tunggu status: `Inactive → Activating → Active` (beberapa menit)
5. Copy **Endpoint URL** dari tab **Endpoint details**

Setelah status `Active`, jalankan Cell 7 untuk test via REST API.

Detail step-by-step + screenshot ada di [`06-model-endpoints.md`](../06-model-endpoints.md#7-step-by-step-aktivasi-endpoint).

## 7. (Optional) Call REST API endpoint

Isi `WORKSPACE_ID`, `MODEL_ID`, dan `VERSION` dari Fabric UI dulu sebelum run.

> Format URL resmi (per [Score ML Model Endpoint Version REST API](https://learn.microsoft.com/en-us/rest/api/fabric/mlmodel/endpoint/score-ml-model-endpoint-version)):
> ```
> https://api.fabric.microsoft.com/v1/workspaces/{wsId}/mlmodels/{modelId}/endpoint/versions/{name}/score
> ```

In [ ]:
import requests
from azure.identity import DefaultAzureCredential

# === ISI INI dari Fabric UI ===
WORKSPACE_ID = "<workspace-id>"   # GUID, copy dari URL Fabric
MODEL_ID     = "<model-id>"       # GUID, copy dari Endpoint details
VERSION      = "1"
# ==============================

# 1. Bearer token (Entra ID)
credential = DefaultAzureCredential()
token = credential.get_token("https://api.fabric.microsoft.com/.default").token

# 2. Endpoint URL
endpoint_url = (
    f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}"
    f"/mlmodels/{MODEL_ID}/endpoint/versions/{VERSION}/score"
)

# 3. Payload — orientation "split" (kolom eksplisit)
payload = {
    "formatType": "dataframe",
    "orientation": "split",
    "inputs": {
        "columns": [
            "sepal length (cm)",
            "sepal width (cm)",
            "petal length (cm)",
            "petal width (cm)",
        ],
        "data": [
            [5.1, 3.5, 1.4, 0.2],
            [6.7, 3.0, 5.2, 2.3],
        ],
    },
}

# 4. Call endpoint
response = requests.post(
    endpoint_url,
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    },
    json=payload,
    timeout=30,
)

print(f"Status code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print("Response:")
    print(result)
elif response.status_code == 202:
    print(f"LRO accepted. Poll: {response.headers.get('Location')}")
    print(f"Retry-After   : {response.headers.get('Retry-After')}s")
elif response.status_code == 429:
    print(f"Throttled. Retry after {response.headers.get('Retry-After')}s")
else:
    print(response.text)

## 8. Cleanup

Setelah selesai eksperimen, **deactivate endpoint** di Fabric UI untuk hentikan konsumsi CU:

- Buka model version → ribbon → **Deactivate version endpoint**
- Atau **Manage endpoints** → pilih beberapa sekaligus → **Deactivate**

Status akan berubah: `Active → Deactivating → Inactive`.

---

## 📚 Referensi

- 📖 [Modul 06 — Tutorial lengkap](../06-model-endpoints.md)
- 📖 [Serve real-time predictions with ML model endpoints (MS Learn)](https://learn.microsoft.com/en-us/fabric/data-science/model-endpoints)
- 📖 [Endpoint REST API](https://learn.microsoft.com/en-us/rest/api/fabric/mlmodel/endpoint)